# Spotify Library — Data Cleaning & Preprocessing

Converts `spotifyLibrary.csv` into a ML-ready dataset for K-Prototypes clustering.

Every decision here is grounded in the findings from **Spotify EDA - Comprehensive.ipynb**.

**Pipeline**
1. Load raw data & audit quality
2. Drop meta / identifier columns
3. Engineer `duration_min`, drop `duration_ms`
4. Remove duplicate rows
5. Drop low-signal features (`loudness`, `time_signature`)
6. Apply `log1p` transforms to right-skewed continuous features
7. `MinMaxScaler` on all continuous features
8. Cast categorical columns to `str` for K-Prototypes
9. Validate & save `spotifyLibrary_cleaned.csv`

# Why K-Prototypes clustering:

K-means(popular clusting technique) uses euclidian distance to assign data points to a cluster calcualted from numerical features only. My dataset has a mix of categorical and numerical data so I will need an algorithm that can leverage both types of features.

K-Protoypes clustering handles both numerical and categorial data. 

---
## 1. Setup & Load

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13, 'axes.labelsize': 11})
COLOR_MAIN = '#1DB954'
COLOR_ACC  = '#191414'

RAW_PATH  = '../data_files/spotifyLibrary.csv'
OUT_PATH  = '../data_files/spotifyLibrary_cleaned.csv'

df_raw = pd.read_csv(RAW_PATH, index_col=0)
print(f'Raw shape: {df_raw.shape}')
df_raw.head(3)

Raw shape: (1573, 18)


,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,type,id,uri,track_href,analysis_url,duration_ms,time_signature
0,0.730,0.677,4,-7.599,0,0.3220,0.0511,0.0,0.0623,0.586,108.695,audio_features,21AJQhGZpujjZQXByZAXpr,spotify:track:21AJQhGZpujjZQXByZAXpr,https://api.spotify.com/v1/tracks/21AJQhGZpujj...,https://api.spotify.com/v1/audio-analysis/21AJ...,190345,3
0,0.801,0.688,1,-4.594,1,0.3960,0.1440,0.0,0.1070,0.500,162.928,audio_features,1MIGkQxcdAt2lDx6ySpsc5,spotify:track:1MIGkQxcdAt2lDx6ySpsc5,https://api.spotify.com/v1/tracks/1MIGkQxcdAt2...,https://api.spotify.com/v1/audio-analysis/1MIG...,157675,4
0,0.597,0.433,10,-10.280,0,0.0444,0.0762,0.0,0.1170,0.159,114.915,audio_features,0xgMhStlvnlNzwewETY4L6,spotify:track:0xgMhStlvnlNzwewETY4L6,https://api.spotify.com/v1/tracks/0xgMhStlvnlN...,https://api.spotify.com/v1/audio-analysis/0xgM...,144000,4


---
## 2. Baseline Quality Audit

Confirm the EDA findings before any transformations: no nulls, 8 duplicate rows, correct dtypes.

In [4]:
print('=== Missing Values ===')
print(df_raw.isnull().sum())
print(f'\nTotal nulls : {df_raw.isnull().sum().sum()}')
print(f'Duplicates  : {df_raw.duplicated().sum()}')
print(f'\nDtypes:')
print(df_raw.dtypes)

=== Missing Values ===
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
type                0
id                  0
uri                 0
track_href          0
analysis_url        0
duration_ms         0
time_signature      0
dtype: int64

Total nulls : 0
Duplicates  : 0

Dtypes:
danceability        float64
energy              float64
key                   int64
loudness            float64
mode                  int64
speechiness         float64
acousticness        float64
instrumentalness    float64
liveness            float64
valence             float64
tempo               float64
type                    str
id                      str
uri                     str
track_href              str
analysis_url            str
duration_ms           int64
time_signature        int64
dtype: object


---
## 3. Drop Identifier / Meta Columns

These columns are API bookkeeping — they carry no audio signal and would corrupt distance calculations.

In [6]:
META_COLS = ['type', 'id', 'uri', 'track_href', 'analysis_url']

df = df_raw.drop(columns=[c for c in META_COLS if c in df_raw.columns])
print(f'Columns after dropping meta: {df.columns.tolist()}')

Columns after dropping meta: ['danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'time_signature']


---
## 4. Engineer `duration_min` & Drop `duration_ms`

`duration_ms` is in the tens of thousands — converting to minutes puts it on a human-interpretable scale and prevents the large raw magnitude from dominating distance metrics before scaling.

In [7]:
df['duration_min'] = df['duration_ms'] / 60_000
df = df.drop(columns=['duration_ms'])

print(f'duration_min  min={df["duration_min"].min():.2f}  '
      f'max={df["duration_min"].max():.2f}  '
      f'mean={df["duration_min"].mean():.2f}')

duration_min  min=0.93  max=19.10  mean=3.65


---
## 5. Remove Duplicate Rows

EDA found 8 exact duplicates. Keeping them would inflate the apparent density of certain audio profiles and bias cluster centroids.

In [8]:
n_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
n_after  = len(df)

print(f'Rows before : {n_before}')
print(f'Rows removed: {n_before - n_after}')
print(f'Rows after  : {n_after}')

Rows before : 1573
Rows removed: 8
Rows after  : 1565


---
## 6. Drop Low-Signal Features

### 6a. `loudness` — redundant with `energy`
EDA showed energy ↔ loudness Pearson r ≈ 0.70. In K-Prototypes (and any distance-based method) keeping both dimensions effectively double-counts the same underlying signal. `energy` is retained because it is already normalised to [0, 1], whereas `loudness` is in negative dB and harder to interpret.

### 6b. `time_signature` — near-zero variance
~95% of tracks are 4/4. A feature that is nearly constant across the dataset contributes almost nothing to cluster separation and can skew K-Prototypes' categorical dissimilarity term.

In [11]:
print('time_signature value counts:')
print(df['time_signature'].value_counts())
print(f'\n4/4 proportion: {(df["time_signature"] == 4).mean():.1%}')

time_signature value counts:
time_signature
4    1489
3      50
5      22
1       4
Name: count, dtype: int64

4/4 proportion: 95.1%


In [12]:
DROP_FEATURES = ['loudness', 'time_signature']
df = df.drop(columns=DROP_FEATURES)

print(f'Remaining columns ({df.shape[1]}): {df.columns.tolist()}')

Remaining columns (11): ['danceability', 'energy', 'key', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_min']


---
## 7. Skewness Transforms for Continuous Features

EDA identified four heavily right-skewed features (|skew| > 0.75). K-Prototypes uses Euclidean distance for continuous dimensions — a spike-shaped distribution inflates distances in that dimension and lets a handful of extreme values dominate cluster assignment.

**`instrumentalness` — binarized instead of transformed**
`instrumentalness` has a bimodal distribution: most tracks sit near 0 (vocal) with a smaller group near 1.0 (fully instrumental). Log transforms cannot fix bimodality. Thresholding at 0.5 turns this into a meaningful binary categorical (`0 = vocal`, `1 = instrumental`) that K-Prototypes routes through its categorical dissimilarity term.

**`liveness` — square root transform**
Square root is gentler than `log1p` and better suited to moderate skew. It compresses the right tail without over-correcting.

**`speechiness`, `duration_min` — `log1p` transform**
`log1p(x)` = `log(1 + x)` handles heavy right tails and exact zeros safely.

| Feature | Raw skew | Transform | Result |
|---|---|---|---|
| `instrumentalness` | 3.996 | binarize (≥ 0.5) | → categorical |
| `liveness` | 1.998 | sqrt | < 1.0 |
| `speechiness` | 1.147 | log1p | < 1.0 |
| `duration_min` | 3.551 | log1p | < 0.5 |

In [ ]:
# Binarize instrumentalness — bimodal distribution, threshold at 0.5
df['instrumentalness'] = (df['instrumentalness'] >= 0.5).astype(int)
print(f'instrumentalness binarized → {df["instrumentalness"].value_counts().to_dict()}')

# Square root transform for liveness (moderate skew)
SQRT_COLS = ['liveness']
# log1p transform for remaining heavy right-skewed features
LOG_COLS  = ['speechiness', 'duration_min']

TRANSFORM_COLS = SQRT_COLS + LOG_COLS
skew_before = {col: round(skew(df[col]), 3) for col in TRANSFORM_COLS}

for col in SQRT_COLS:
    df[col] = np.sqrt(df[col])

for col in LOG_COLS:
    df[col] = np.log1p(df[col])

skew_after = {col: round(skew(df[col]), 3) for col in TRANSFORM_COLS}

skew_report = pd.DataFrame({'skew_before': skew_before, 'skew_after': skew_after}).T
print('\nSkewness before and after transforms:')
skew_report

In [ ]:
# Visual confirmation — before/after distributions for each transformed feature
df_pre = pd.read_csv(RAW_PATH, index_col=0).drop_duplicates().reset_index(drop=True)
df_pre['duration_min'] = df_pre['duration_ms'] / 60_000

fig, axes = plt.subplots(len(TRANSFORM_COLS), 2, figsize=(12, 3 * len(TRANSFORM_COLS)),
                          constrained_layout=True)

for row, col in enumerate(TRANSFORM_COLS):
    raw   = df_pre[col].dropna()
    trans = df[col].dropna()
    label = 'sqrt' if col in SQRT_COLS else 'log1p'

    for ax, data, title in zip(
        axes[row],
        [raw, trans],
        [f'{col}  (raw, skew={skew(raw):.2f})',
         f'{col}  ({label}, skew={skew(trans):.2f})']):

        ax.hist(data, bins=40, color=COLOR_MAIN, alpha=0.55,
                edgecolor='white', linewidth=0.4, density=True)
        data.plot.kde(ax=ax, color=COLOR_ACC, linewidth=2)
        ax.set_title(title, fontsize=10)
        ax.set_ylabel('Density')

fig.suptitle('Skew Transforms — Before vs After', fontsize=14, fontweight='bold')
plt.show()

# Instrumentalness binarization
fig, axes = plt.subplots(1, 2, figsize=(10, 3), constrained_layout=True)

df_pre['instrumentalness'].hist(bins=40, ax=axes[0], color=COLOR_MAIN,
                                alpha=0.55, edgecolor='white', linewidth=0.4)
axes[0].set_title(f'instrumentalness  (raw, skew={skew(df_pre["instrumentalness"]):.2f})', fontsize=10)
axes[0].set_ylabel('Count')

df['instrumentalness'].value_counts().sort_index().plot.bar(
    ax=axes[1], color=COLOR_MAIN, alpha=0.7, edgecolor='white')
axes[1].set_title('instrumentalness  (binarized: 0=vocal, 1=instrumental)', fontsize=10)
axes[1].set_ylabel('Count')
axes[1].set_xticklabels(['0 (vocal)', '1 (instrumental)'], rotation=0)

fig.suptitle('Instrumentalness — Raw Distribution vs Binarized', fontsize=14, fontweight='bold')
plt.show()

---
## 8. MinMax Scale Continuous Features

All continuous features are scaled to [0, 1] so no single feature dominates K-Prototypes' Euclidean distance term due to a wider numeric range (e.g. `tempo` is in the hundreds while `danceability` is 0–1).

**Categorical columns (`key`, `mode`) are deliberately excluded** — they will be handled by K-Prototypes' categorical dissimilarity term and must remain as discrete labels.

In [ ]:
CAT_COLS  = ['key', 'mode', 'instrumentalness']
CONT_COLS = [c for c in df.columns if c not in CAT_COLS]

print(f'Continuous features to scale ({len(CONT_COLS)}): {CONT_COLS}')
print(f'Categorical features kept as-is ({len(CAT_COLS)}): {CAT_COLS}')

In [ ]:
scaler = MinMaxScaler()
df[CONT_COLS] = scaler.fit_transform(df[CONT_COLS])

print('Continuous feature ranges after scaling:')
df[CONT_COLS].agg(['min', 'max']).T

---
## 9. Cast Categorical Columns to String

K-Prototypes expects categorical columns to be non-numeric so it routes them through the Hamming-style dissimilarity term instead of Euclidean distance. Casting to `str` satisfies this requirement without creating ordinal assumptions.

In [ ]:
for col in CAT_COLS:
    df[col] = df[col].astype(str)

print('Dtypes after casting:')
print(df.dtypes)

---
## 10. Validation

Final checks before saving: no nulls, correct shape, all continuous features in [0, 1], categorical columns are strings.

In [ ]:
print('=== Final Dataset Shape ===')
print(f'{df.shape[0]:,} tracks  ×  {df.shape[1]} features')
print()

print('=== Null Check ===')
null_total = df.isnull().sum().sum()
print(f'Total nulls: {null_total}', '✓' if null_total == 0 else '✗ FIX REQUIRED')
print()

print('=== Continuous Feature Range Check (all should be [0.0, 1.0]) ===')
range_check = df[CONT_COLS].agg(['min', 'max']).T
range_check['in_range'] = (range_check['min'] >= 0) & (range_check['max'] <= 1)
print(range_check.to_string())
print()

print('=== Categorical Column Dtypes ===')
for col in CAT_COLS:
    print(f'  {col}: dtype={df[col].dtype}  unique={df[col].nunique()} values')

In [ ]:
# Distribution overview of the cleaned, scaled continuous features
fig, axes = plt.subplots(2, 4, figsize=(18, 8), constrained_layout=True)
axes = axes.flatten()

for i, col in enumerate(CONT_COLS):
    ax = axes[i]
    data = df[col]
    ax.hist(data, bins=40, color=COLOR_MAIN, alpha=0.55,
            edgecolor='white', linewidth=0.4, density=True)
    data.plot.kde(ax=ax, color=COLOR_ACC, linewidth=2)
    ax.axvline(data.mean(), color='#e74c3c', linestyle='--',
               linewidth=1.4, label=f'mean={data.mean():.2f}')
    ax.set_title(col)
    ax.set_xlim(0, 1)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Cleaned & Scaled Feature Distributions', fontsize=15, fontweight='bold')
plt.show()

In [ ]:
# Correlation heatmap on cleaned data — confirm energy/acousticness split still intact
fig, ax = plt.subplots(figsize=(9, 7))
corr = df[CONT_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=-1, vmax=1, center=0, square=True, linewidths=0.5,
            linecolor='white', ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Cleaned Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 11. Save Cleaned Dataset

In [ ]:
df.to_csv(OUT_PATH, index=False)
print(f'Saved: {OUT_PATH}')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

---
## 12. Cleaning Summary

| Step | Action | Rationale |
|---|---|---|
| Meta drop | Removed `type`, `id`, `uri`, `track_href`, `analysis_url` | No audio signal; would corrupt distance metrics |
| Feature engineer | `duration_ms` → `duration_min` | Human-interpretable; same relative scale |
| Deduplication | Removed 8 exact duplicate rows | Prevents centroid bias |
| Feature drop | Removed `loudness` | Energy ↔ Loudness r ≈ 0.70; energy retained as already-bounded [0,1] |
| Feature drop | Removed `time_signature` | ~95% 4/4; near-zero variance adds noise, not signal |
| Binarize | `instrumentalness` → 0/1 (threshold 0.5) | Bimodal distribution — log transform ineffective; binary categorical treated via K-Prototypes dissimilarity term |
| sqrt transform | `liveness` | Moderate skew corrected with gentler transform |
| log1p transform | `speechiness`, `duration_min` | Corrects heavy right skew before distance-based clustering |
| MinMax scale | All 7 continuous features → [0, 1] | Prevents wide-range features from dominating Euclidean distance |
| Categorical cast | `key`, `mode`, `instrumentalness` → `str` | Satisfies K-Prototypes' categorical dissimilarity routing |

**Output:** `spotifyLibrary_cleaned.csv` — 7 scaled continuous features + 3 categorical features, ready for K-Prototypes with `n_clusters` sweep 5–9.